# Scale hints: MapAnything teacher (E3)

Run on a **different** machine/session from `vlm.ipynb` (Colab T4 is fine, or a second Kaggle notebook). Internet on.
T4 is Turing like the RTX 2060: weights run in **bf16** (fp16 gives all-NaN outputs; see `teacher_patches.py`).

Output: `cache/teacher_scale.json`. Download it, commit it to the repo, push; then `git pull` in the VLM notebook.

In [ ]:
import os, sys, subprocess
REPO_URL = 'https://github.com/quagmire02/gsrvlm.git'   # <- edit
ON_KAGGLE = os.path.exists('/kaggle/working')
ON_COLAB = 'google.colab' in sys.modules
WORK = '/kaggle/working' if ON_KAGGLE else '/content'
if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive'
os.chdir(WORK)
if not os.path.exists('gsrvlm'):
    subprocess.run(['git', 'clone', REPO_URL, 'gsrvlm'], check=True)
os.chdir('gsrvlm')
!git pull
!nvidia-smi -L

In [ ]:
!pip install -q -r requirements.txt
!bash install_teacher.sh

In [ ]:
# same deterministic frames as the VLM machine
!python prepare_data.py --download_dir /tmp/vsi

In [ ]:
# smoke test on one GPU first; prints s_med for the first scene
!python run_teacher.py --nshards 60 --shard 0

In [ ]:
import torch
os.makedirs('logs', exist_ok=True)
n = torch.cuda.device_count()
procs = [subprocess.Popen(f'CUDA_VISIBLE_DEVICES={g} python run_teacher.py --shard {g} --nshards {n} > logs/teacher.s{g}.log 2>&1', shell=True) for g in range(n)]
print([p.wait() for p in procs])
!tail -n 5 logs/teacher.s*.log
!python run_teacher.py --merge

Now download `cache/teacher_scale.json` (Files pane / Kaggle output), commit it to the repo and push.
Note: the smoke-test cell wrote `cache/teacher_scale.s0.json` with nshards=60 sharding; that is fine, `--merge` combines every shard file.